# Analítica de Recursos Humanos: Determinantes Salariales y Modelación de Compensaciones
**ICS40125 — Analítica para Negocios**

Integrantes: Angello Avellaneda, Vicente Yañez, Francisco Cariga, Alvaro Vergara, Sebastian Rodriguez

# Parte 1: Análisis Exploratorio de Datos (EDA)

In [ ]:
# PARTE 1 — LIMPIEZA DE DATOS
import pandas as pd
import numpy as np

# Cargar datos
path = 'https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/projects/data/employee_compensation.csv'
df = pd.read_csv(path, sep=",")

# Inspección inicial: primeras 10 filas, tipos de datos y duplicados
display(df.head(10))
df.info()
print(f"\nRegistros duplicados detectados: {df.duplicated().sum()}")

# 0. Eliminar duplicados exactos: 97 filas idénticas en las 9 columnas
#    (incluido Salary y Performance_Score) -> se tratan como artefactos de registro.
df = df.drop_duplicates()

# 1. Normalizar TODAS las categóricas (incluido Job_Level) ANTES de imputar.
#    .str.strip().str.title() unifica capitalización y espacios.
#    Los strings 'nan'/' nan' del CSV se convierten a NaN reales para que la
#    imputación los alcance (si no, generan categorías falsas tipo 'Nan' al codificar).
cols_categoricas = ['Education_Level', 'Gender', 'Department', 'Region', 'Job_Level']
for col in cols_categoricas:
    df[col] = df[col].str.strip().str.title()
    df[col] = df[col].replace({'Nan': np.nan})

# 2. Eliminar filas con nulos en columnas críticas (variable objetivo y desempeño)
df = df.dropna(subset=['Salary', 'Performance_Score'])

# 3. Imputar variables categóricas con la moda
for col in cols_categoricas:
    df[col] = df[col].fillna(df[col].mode()[0])

# 4. Imputar variables numéricas con la mediana (robusta a outliers)
for col in ['Years_Experience', 'Age']:
    df[col] = df[col].fillna(df[col].median())

# 5. Recorte de outliers a límites lógicos del contexto laboral
n_xp_out  = ((df['Years_Experience'] < 0) | (df['Years_Experience'] > 50)).sum()
n_age_out = ((df['Age'] < 18) | (df['Age'] > 65)).sum()
df['Years_Experience'] = df['Years_Experience'].clip(lower=0, upper=50)
df['Age'] = df['Age'].clip(lower=18, upper=65)

# --- Reporte de limpieza ---
print('\n### Reporte de Limpieza de Datos ###')
print('Valores nulos por columna después de la limpieza:')
display(df.isnull().sum())
print(f"\nDimensiones finales: {df.shape[0]} filas y {df.shape[1]} columnas")
print(f"Outliers ajustados -> Years_Experience: {n_xp_out} | Age: {n_age_out}")

print('\n### Estadísticas Descriptivas ###')
display(df[['Salary', 'Years_Experience', 'Age', 'Performance_Score']].describe().round(2))

,Salary,Years_Experience,Age,Performance_Score,Education_Level,Department,Region,Job_Level,Gender
0,69181.0,3.0,22.0,3.5,Bachelor,Sales,East,Junior,Male
1,88466.0,5.0,23.0,2.8,bachelor,FINANCE,West,Junior,Female
2,76112.0,11.0,29.0,1.6,Master,HR,West,Mid,Female
3,NaN,15.0,37.0,NaN,Bachelor,Marketing,South,Senior,Male
4,NaN,6.0,28.0,4.5,Bachelor,Engineering,North,Mid,Female
5,81246.0,4.0,21.0,2.8,Bachelor,Engineering,West,Junior,Male
6,71494.0,4.0,23.0,2.4,Bachelor,Engineering,South,Junior,Male
7,78189.0,3.0,21.0,3.5,Master,Finance,East,Junior,NaN
8,67204.0,0.0,21.0,3.5,Master,Marketing,East,Junior,Female
9,59776.0,4.0,24.0,NaN,Bachelor,HR,East,Junior,Female


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10100 entries, 0 to 10099
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Salary             9897 non-null   float64
 1   Years_Experience   9695 non-null   float64
 2   Age                9797 non-null   float64
 3   Performance_Score  9497 non-null   float64
 4   Education_Level    9594 non-null   object 
 5   Department         9795 non-null   object 
 6   Region             9699 non-null   object 
 7   Job_Level          10100 non-null  object 
 8   Gender             9599 non-null   object 
dtypes: float64(4), object(5)
memory usage: 710.3+ KB

Registros duplicados detectados: 97

### Reporte de Limpieza de Datos ###
Valores nulos por columna después de la limpieza:


,0
Salary,0
Years_Experience,0
Age,0
Performance_Score,0
Education_Level,0
Department,0
Region,0
Job_Level,0
Gender,0



Dimensiones finales: 9216 filas y 9 columnas
Outliers ajustados -> Years_Experience: 26 | Age: 37

### Estadísticas Descriptivas ###


,Salary,Years_Experience,Age,Performance_Score
count,9216.00,9216.00,9216.00,9216.00
mean,95768.43,6.79,29.42,3.03
std,30594.08,5.36,6.01,1.83
min,5342.00,0.00,18.00,-1.00
25%,76898.50,3.00,25.00,2.00
50%,91726.00,6.00,29.00,3.00
75%,112004.25,10.00,33.00,4.00
max,394249.00,50.00,65.00,99.00


In [ ]:
# PARTE 1 — VISUALIZACIONES
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# 1. Histograma del salario
sns.histplot(df['Salary'], kde=True, bins=40, color='#2c3e50', ax=axes[0])
axes[0].axvline(df['Salary'].mean(),   color='red',   linestyle='--', label=f"Media: ${df['Salary'].mean():,.0f}")
axes[0].axvline(df['Salary'].median(), color='green', linestyle='-',  label=f"Mediana: ${df['Salary'].median():,.0f}")
axes[0].set_title('Distribución Salarial Anual', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Salario (USD)'); axes[0].legend()

# 2. Boxenplot del salario por nivel de cargo
sns.boxenplot(x='Job_Level', y='Salary', data=df, order=['Junior', 'Mid', 'Senior'],
              hue='Job_Level', palette='viridis', legend=False, ax=axes[1])
axes[1].set_title('Salario por Nivel de Cargo', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Nivel de Cargo'); axes[1].set_ylabel('Salario Anual (USD)')

# 3. Dispersión experiencia vs salario con línea de regresión
sns.regplot(x='Years_Experience', y='Salary', data=df,
            scatter_kws={'alpha':0.3, 's':15, 'color':'#e67e22'},
            line_kws={'color':'#d35400', 'lw':3}, ax=axes[2])
axes[2].set_title('Tendencia: Experiencia vs. Salario', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Años de Experiencia'); axes[2].set_ylabel('Salario Anual (USD)')

plt.tight_layout(); plt.show()

# 4. Mapa de calor de correlaciones (variables numéricas)
df_numeric = df.select_dtypes(include=[np.number])
corr = df_numeric.corr()
plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap='coolwarm',
            linewidths=0.5, cbar_kws={"shrink": .8})
plt.title('Mapa de Calor: Correlación entre Variables Numéricas', fontsize=15, pad=20)
plt.xticks(rotation=45); plt.show()

# Parte 2: Pruebas de Hipótesis

In [ ]:
# PARTE 2 — PRUEBAS DE HIPÓTESIS
from scipy import stats

# --- 1. Años de Experiencia vs. Salario ---
# H0: no hay relación lineal (rho = 0)   |   H1: asociación positiva (rho > 0)
coef_pearson, p_pearson = stats.pearsonr(df['Years_Experience'], df['Salary'])
print("### Años de Experiencia vs. Salario ###")
print(f"Coeficiente de Pearson: {coef_pearson:.4f}")
print(f"P-valor: {p_pearson:.4e}")
alfa = 0.05
if p_pearson < alfa:
    print("Se rechaza H0: la experiencia se asocia positiva y significativamente con el salario.")
else:
    print("No se rechaza H0.")

print("-" * 40)

# --- 2. (Bonus) Junior vs. Senior — prueba t de Welch ---
# H0: medias iguales   |   H1: medias distintas
salarios_junior = df[df['Job_Level'] == 'Junior']['Salary']
salarios_senior = df[df['Job_Level'] == 'Senior']['Salary']
print("### Diferencia Salarial Junior vs. Senior ###")
print(f"Media Junior: ${salarios_junior.mean():,.0f}  (n={len(salarios_junior)})")
print(f"Media Senior: ${salarios_senior.mean():,.0f}  (n={len(salarios_senior)})")
t_stat, p_ttest = stats.ttest_ind(salarios_junior, salarios_senior, equal_var=False)
print(f"Estadístico t: {t_stat:.4f}")
print(f"P-valor: {p_ttest:.4e}")
if p_ttest < alfa:
    print("Se rechaza H0: la diferencia salarial entre Junior y Senior es significativa.")
else:
    print("No se rechaza H0.")

NameError: name 'df' is not defined

# Parte 3: Regresión Lineal con Variables Categóricas

## Paso 1: Justificación de variables (antes de codificar)

**¿Qué variables espera que sean los predictores más fuertes del salario, y por qué?**

- **Job_Level**: es el predictor más fuerte esperado. La prueba t de la Parte 2 confirmó diferencias salariales enormes entre Junior y Senior (t ≈ −79,7). Las bandas salariales se estructuran principalmente en torno al nivel de cargo.
- **Years_Experience**: correlación de Pearson r ≈ 0,59 con el salario, la más alta entre las variables numéricas. A mayor experiencia, mayor productividad esperada y mayor remuneración.
- **Department**: distintas áreas tienen mercados laborales y bandas salariales sistemáticamente diferentes.

**¿Hay variables que incluiría aunque sean estadísticamente insignificantes? ¿Por qué?**

- **Gender**: aunque su efecto sea pequeño, debe incluirse para detectar posibles brechas salariales de género. Excluirla ocultaría un sesgo relevante desde la equidad y RRHH.
- **Region**: los costos de vida y mercados laborales varían geográficamente; omitirla podría sesgar predicciones por localización.

**¿Anticipa variables fuertemente correlacionadas entre sí? ¿Qué problema causaría?**

El par más evidente es **Age y Years_Experience** (a mayor edad, más experiencia acumulada típicamente). También puede haber correlación entre Job_Level y Years_Experience. Esto produce multicolinealidad, que infla los errores estándar de los coeficientes y dificulta interpretar el efecto aislado de cada variable, aunque no invalida el poder predictivo global del modelo.

In [ ]:
# PARTE 3, PASO 2 — Preparación de los datos
# Trabajamos sobre una COPIA para no alterar `df` (así la Parte 2 sigue siendo
# reejecutable aunque corramos esta celda después).
df_model = df.copy()

# Label Encoding para Job_Level (ordinal: Junior < Mid < Senior)
orden_job_level = {'Junior': 0, 'Mid': 1, 'Senior': 2}
df_model['Job_Level'] = df_model['Job_Level'].map(orden_job_level)

# One-Hot Encoding (drop_first=True) para las variables nominales
df_model = pd.get_dummies(
    df_model,
    columns=['Education_Level', 'Department', 'Region', 'Gender'],
    drop_first=True
)

print(df_model['Job_Level'].value_counts().sort_index())
print(f"\nDimensiones: {df_model.shape}")
print(f"\nColumnas:\n{list(df_model.columns)}")

In [ ]:
# PARTE 3, PASO 3 — Construcción del modelo
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm

X = df_model.drop(columns=['Salary']).astype(float)
y = df_model['Salary'].astype(float)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# OLS con statsmodels
X_train_sm = sm.add_constant(X_train)
modelo_ols = sm.OLS(y_train, X_train_sm).fit()
print(modelo_ols.summary())

# Regresión lineal con scikit-learn
modelo_sklearn = LinearRegression().fit(X_train, y_train)
print("\nModelo sklearn entrenado correctamente.")

In [ ]:
# PARTE 3, PASO 4 — Evaluación del modelo
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X_test_sm = sm.add_constant(X_test, has_constant='add')
assert list(X_train_sm.columns) == list(X_test_sm.columns), "Columnas train/test no coinciden"

def evaluar_modelo(y_real, y_pred, nombre, n, p):
    mae  = mean_absolute_error(y_real, y_pred)
    mape = (abs((y_real - y_pred) / y_real)).mean() * 100
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    r2   = r2_score(y_real, y_pred)
    r2_adj = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    print(f"\n### {nombre} ###")
    print(f"MAE:         {mae:,.2f}")
    print(f"MAPE:        {mape:.2f}%")
    print(f"RMSE:        {rmse:,.2f}")
    print(f"R²:          {r2:.4f}")
    print(f"R² ajustado: {r2_adj:.4f}")

p = X_train.shape[1]
n_tr, n_te = len(y_train), len(y_test)

evaluar_modelo(y_train, modelo_sklearn.predict(X_train), "Sklearn - Entrenamiento", n_tr, p)
evaluar_modelo(y_test,  modelo_sklearn.predict(X_test),  "Sklearn - Prueba",        n_te, p)
evaluar_modelo(y_train, modelo_ols.predict(X_train_sm),  "OLS - Entrenamiento",     n_tr, p)
evaluar_modelo(y_test,  modelo_ols.predict(X_test_sm),   "OLS - Prueba",            n_te, p)
print(f"\nCondition number (OLS): {modelo_ols.condition_number:.2e}")

# Parte 4: Análisis y Refinamiento del Modelo

## Paso 1: Interpretación de los p-valores de los coeficientes

In [ ]:
# Tabla de p-valores ordenados
pvalores = modelo_ols.pvalues.drop('const').sort_values(ascending=False)
tabla_pval = pvalores.to_frame(name='p-valor').reset_index()
tabla_pval.columns = ['Variable', 'p-valor']
tabla_pval['Significativa (5%)'] = tabla_pval['p-valor'].apply(lambda x: 'Sí' if x < 0.05 else 'No')
display(tabla_pval.round(4))

### Análisis de variables con p-valor > 0,05

Tras la limpieza corregida, la **única** variable con p-valor > 0,05 es **`Region_West`** (p ≈ 0,058, marginalmente insignificante).

**Decisión: se conserva.** La variabilidad salarial geográfica es relevante para TalentCo, y su insignificancia marginal puede deberse a que parte del efecto regional ya está capturado por otras variables. Eliminarla podría sesgar las predicciones para empleados de esa región y aportaría un beneficio nulo en términos de simplicidad. Como verificación, abajo comparamos el modelo completo con una alternativa sin `Region_West`: las métricas son prácticamente idénticas, lo que confirma que conservarla no tiene costo predictivo.

> Nota: con la limpieza corregida ya **no aparecen** columnas residuales tipo `Region_Nan`/`Gender_Nan` (los strings `'nan'` del CSV se convirtieron a nulos reales y se imputaron). Por eso el modelo final conserva las 14 variables predictoras, todas con interpretación de negocio.

In [ ]:
# Comparación: modelo completo vs. alternativa sin Region_West
X_train_alt = X_train.drop(columns=['Region_West'])
X_test_alt  = X_test.drop(columns=['Region_West'])
modelo_alt = sm.OLS(y_train, sm.add_constant(X_train_alt)).fit()

print("--- Modelo completo (FINAL: 14 predictores) ---")
evaluar_modelo(y_test, modelo_ols.predict(X_test_sm), "OLS Completo - Prueba", n_te, X_train.shape[1])

print("\n--- Alternativa sin Region_West (descartada) ---")
evaluar_modelo(y_test, modelo_alt.predict(sm.add_constant(X_test_alt, has_constant='add')),
               "OLS sin Region_West - Prueba", n_te, X_train_alt.shape[1])

# El modelo final es el completo
modelo_final = modelo_ols
X_train_final_sm = X_train_sm

## Paso 2: Análisis de Multicolinealidad (VIF)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# VIF calculado CON la constante incluida (procedimiento correcto: si se omite el
# intercepto, los VIF de variables con media != 0 se inflan artificialmente).
X_vif = sm.add_constant(X_train)
vif_data = pd.DataFrame({
    'Variable': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})
vif_data = vif_data[vif_data['Variable'] != 'const'].sort_values('VIF', ascending=False).reset_index(drop=True)
display(vif_data.round(2))

# Visualización
plt.figure(figsize=(10, 6))
colores = ['#e74c3c' if v > 10 else '#e67e22' if v > 5 else '#2ecc71' for v in vif_data['VIF']]
plt.barh(vif_data['Variable'], vif_data['VIF'], color=colores)
plt.axvline(x=10, color='red',    linestyle='--', linewidth=1, label='Umbral severo (10)')
plt.axvline(x=5,  color='orange', linestyle='--', linewidth=1, label='Umbral moderado (5)')
plt.gca().invert_yaxis()
plt.xlabel('VIF'); plt.title('Factor de Inflación de Varianza por Variable', fontsize=13, fontweight='bold')
plt.legend(); plt.tight_layout(); plt.show()

### Interpretación del VIF

Calculado correctamente (con constante), **ningún VIF supera el umbral severo de 10**: `Job_Level` ≈ 5,5, `Years_Experience` ≈ 3,6 y `Age` ≈ 3,1 son los más altos; el resto queda por debajo de 2. Esto refleja la correlación real entre edad y experiencia (r ≈ 0,72 en el heatmap), pero la multicolinealidad es **moderada, no severa**.

**Decisión: se conservan Age y Years_Experience.** Capturan dimensiones conceptualmente distintas —edad biológica vs. trayectoria profesional acumulada— y su VIF moderado no compromete el modelo. La multicolinealidad afecta principalmente la precisión de los coeficientes individuales, no el poder predictivo global, por lo que es una limitación menor y aceptable para el objetivo del análisis.

## Paso 3: Verificación de Supuestos de Regresión

In [ ]:
import scipy.stats as stats
from statsmodels.stats.diagnostic import het_breuschpagan

residuos = y_train - modelo_final.predict(X_train_final_sm)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(residuos, kde=True, bins=40, color='#2c3e50', ax=axes[0])
axes[0].axvline(0, color='red', linestyle='--', linewidth=1.5, label='Residuo = 0')
axes[0].set_title('Distribución de Residuos', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Residuo (USD)'); axes[0].set_ylabel('Frecuencia'); axes[0].legend()
stats.probplot(residuos, dist="norm", plot=axes[1])
axes[1].set_title('Gráfico Q-Q de Residuos', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Breusch-Pagan (homocedasticidad)
bp_lm, bp_pvalue, _, _ = het_breuschpagan(residuos, X_train_final_sm)
print("### Prueba de Breusch-Pagan — Homocedasticidad ###")
print(f"Estadístico LM: {bp_lm:.4f}")
print(f"P-valor:        {bp_pvalue:.4e}")
if bp_pvalue < 0.05:
    print("Se rechaza H0 -> heterocedasticidad. Mitigación: errores estándar robustos (HC3).")
else:
    print("No se rechaza H0 -> homocedasticidad razonable.")

# Parte 5: Informe de Negocio

In [ ]:
# Cálculo de apoyo para el informe: aumento esperado al ascender de Mid a Senior.
# Job_Level está codificado 0=Junior, 1=Mid, 2=Senior. Un ascenso Mid->Senior = +1 paso,
# por lo que el aumento esperado equivale al coeficiente de Job_Level.
coef_job = modelo_final.params['Job_Level']
coef_exp = modelo_final.params['Years_Experience']
coef_gen = modelo_final.params['Gender_Male']
print(f"Aumento esperado Mid -> Senior:        ${coef_job:,.0f} anuales")
print(f"Efecto de +1 año de experiencia:       ${coef_exp:,.0f} anuales")
print(f"Brecha de género (hombres vs mujeres): ${coef_gen:,.0f} anuales")

## Informe Ejecutivo — Determinantes Salariales en TalentCo

**Para:** Dirección de Recursos Humanos de TalentCo

Tras analizar los registros de compensación de más de 9.200 empleados, presentamos los principales hallazgos sobre cómo se determinan los salarios en TalentCo y recomendaciones para fortalecer la equidad y la transparencia de su política de compensaciones.

**Principales determinantes del salario.** El factor que más influye en la compensación es el **nivel de cargo** (Junior, Mid, Senior), seguido por el **departamento** y el **nivel educacional**. La experiencia profesional y el desempeño también aportan, aunque en menor magnitud. En conjunto, estos factores explican una parte sustancial de las diferencias salariales, lo que indica que la estructura responde mayoritariamente a criterios objetivos y razonables.

**Impacto de la experiencia.** Cada año adicional de experiencia profesional se asocia, en promedio, con un incremento cercano a **$830 USD anuales**, manteniendo el resto de factores constante. Es un efecto real y consistente, aunque más moderado que el del nivel de cargo.

**Diferencias por departamento y región.** Existen brechas claras entre departamentos: **Ingeniería** es el área mejor remunerada, mientras que Recursos Humanos, Ventas y Marketing perciben en promedio entre **$13.000 y $18.000 USD menos al año** en perfiles equivalentes. Las diferencias por región son mucho menores (entre $2.000 y $3.000 USD), por lo que la geografía no parece ser una fuente importante de inequidad.

**Ascenso de Mid a Senior.** Según el modelo, promover a un empleado de nivel Mid a Senior se asocia con un aumento esperado de aproximadamente **$19.200 USD anuales**, manteniendo todo lo demás constante. Esta cifra es útil para presupuestar promociones y comunicar trayectorias de carrera.

**Recomendaciones.**
1. Revisar las brechas entre departamentos para confirmar que reflejan diferencias reales de mercado y no inequidades internas.
2. Auditar la **diferencia salarial por género** detectada (los hombres perciben en promedio unos **$1.370 USD más** que las mujeres en perfiles comparables): aunque modesta, es estadísticamente real y merece seguimiento.
3. Formalizar **bandas salariales** por nivel y experiencia para dar previsibilidad.
4. Publicar **criterios de promoción** claros para reforzar la percepción de justicia.

En síntesis, la compensación en TalentCo está mayormente bien estructurada, pero existen oportunidades concretas para mejorar la equidad de género y la transparencia entre departamentos.

## Cierre — Partes 1 a 5

Notebook completo y reproducible: limpieza de datos, EDA, pruebas de hipótesis, regresión OLS y scikit-learn, refinamiento (p-valores, VIF, supuestos) e informe ejecutivo.